# Extracción de metadatos de estudios
Este notebook lee imágenes de diferentes estudios y extrae sus respectivos metadatos para exportarlos en un archivo de Excel

Elaborado por Juan Manuel Rivera/AJS

# Importación de librerías y funciones auxiliares

In [21]:
import os
import SimpleITK as sitk
import numpy as np
import pandas as pd

En la siguiente celda está la dirección del directorio donde están las imágenes

In [22]:
paths = [
    ("AISD","D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset001_AISD"),
    ("ISLES", "D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset003_ISLES"),
    ("APIS", "D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset005_APIS"),
    ("FSFB","D:/jm.rivera/nnUnet_data/nnUNet_raw/Dataset007_FSFB1C")
    ]

# Extracción de metadatos

Si desea añadir algún metadato añada el DICOM Tag en la siguiente lista

In [23]:
tags = {
    #DB
    "db" : [],
    # File path
    "file_path" : [],
    # Mean z-spacing
    "spacing" : [],
    # Number of slices
    "size" : [],
    # Lesion total size
    "lesion_size":[],
    # Number of conected lesions
    "connected_components" : [],
    # Mean component size
    "connected_size" : [],
    # SD component size
    "connected_sd" : [],
    "connected_sizes" : []
}

In [24]:
for datasets in paths:
    for img_path in os.listdir(os.path.join(datasets[1], "labelsTr")):
        seg = os.path.join(datasets[1], "labelsTr",img_path)
        vol = os.path.join(datasets[1], "imagesTr",f"{img_path[:-7]}_0000{img_path[-7:]}")

        original_img = sitk.ReadImage(vol)
        seg_img = sitk.ReadImage(seg)

        seg_array = sitk.GetArrayFromImage(seg_img)

        cc = sitk.ConnectedComponent(seg_img, True)
        stats = sitk.LabelShapeStatisticsImageFilter()
        stats.Execute(cc)

        spacing = original_img.GetSpacing()
        voxel_volume = spacing[0] * spacing[1] * spacing[2]

        for tag, array in tags.items():
            if tag == "db":
                array.append(datasets[0])
            if tag == 'file_path':
                array.append(img_path)
            elif tag == 'spacing':
                array.append(spacing)
            elif tag == 'size':
                array.append(original_img.GetSize())
            elif tag == 'lesion_size':
                array.append( np.count_nonzero(seg_array)*voxel_volume)
            elif tag == "connected_components":
                array.append(stats.GetNumberOfLabels())
            elif tag == "connected_sizes" or tag == "connected_size" or tag == "connected_sd":
                sizes = [stats.GetNumberOfPixels(label) * voxel_volume for label in stats.GetLabels()]
                if len(sizes) == 0:
                    array.append(0)
                else:
                    sizes = np.array(sizes)
                    if tag == "connected_size":
                        array.append(sizes.mean())
                    elif tag == "connected_sd":
                        array.append(sizes.std())
                    else:
                        array.append(sizes)

In [25]:
df = pd.DataFrame(data=tags)
display(df.head(5))

,db,file_path,spacing,size,lesion_size,connected_components,connected_size,connected_sd,connected_sizes
0,AISD,AISD_001.nii.gz,"(0.4375, 0.4375, 8.0)","(512, 512, 17)",29231.562500,1,29231.562500,0.000000,[29231.5625]
1,AISD,AISD_002.nii.gz,"(0.4296875, 0.4296875, 9.26413345336914)","(512, 512, 16)",7823.595759,9,869.288418,1208.832648,"[259.98831554315984, 424.19146220199764, 95.78..."
2,AISD,AISD_003.nii.gz,"(0.404296875, 0.404296875, 3.0044877529144287)","(512, 512, 42)",23663.231755,3,7887.743918,10617.222990,"[22902.515625450655, 455.25103430455147, 305.4..."
3,AISD,AISD_004.nii.gz,"(0.50390625, 0.50390625, 9.002614974975586)","(512, 512, 14)",3972.994270,1,3972.994270,0.000000,[3972.994269682502]
4,AISD,AISD_005.nii.gz,"(0.4375, 0.4375, 8.0)","(512, 512, 19)",30957.281250,8,3869.660156,7135.597950,"[940.1875, 499.1875, 22388.40625, 4694.8125, 6..."


In [26]:
df.describe()

,lesion_size,connected_components,connected_size,connected_sd
count,771.000000,771.000000,771.000000,771.000000
mean,34593.924121,5.382620,15528.647938,9050.230444
std,57563.943929,7.466299,40518.727195,20105.536433
min,0.000000,0.000000,0.000000,0.000000
25%,3555.062218,1.000000,772.214583,0.000000
50%,12874.815958,3.000000,2973.915257,1562.752862
75%,38628.838516,6.000000,11269.120602,7557.147819
max,384269.705347,84.000000,384269.705347,163920.748904


In [27]:
df.to_excel("MetadatosNIfTI.xlsx")